# Femur Model Ablation Study: Heatmap vs Coordinate Regression

Evaluates 3 architectural approaches on the official FP+UCL Femur test set (N=348 images / 266 patients):
- **Row A**: Heatmap landmarks (U-Net EfficientNet-B4, 384px letterbox, 1-channel dual-peak, flip TTA)
- **Row B**: Direct coordinate regression (ConvNeXt-Tiny, 384px letterbox, order-invariant loss)
- **Row C**: Heatmap landmarks (U-Net EfficientNet-B4, 512px letterbox, 1-channel dual-peak, flip TTA)

Each row trained with 3 random seeds (42, 43, 44). Early stopping on 65 patient-grouped validation images.
Patient-level bootstrap 95% confidence intervals computed with 1,000 resamples.

In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

results_dir = Path('/kaggle/working/out')
if not results_dir.is_dir():
    results_dir = Path('../../docs/results/femur_ablation')
if not results_dir.is_dir():
    results_dir = Path('docs/results/femur_ablation')

table_path = results_dir / 'femur_ablation_table.md'
if table_path.is_file():
    display(Markdown(table_path.read_text(encoding='utf-8')))
else:
    print('Table not found at', table_path)

# Femur Model Ablation Study (FP + UCL Femur Test Set, N=348 images / 266 patients)

> Reference FL: Euclidean distance between annotated endpoints * px_to_mm_rate.
> Protocol: Train on 451 images, early stop on 65 validation images (0 patient overlap).
> Evaluation: 348 official test images (323 FP / 261 patients, 25 UCL / 5 patients [tiny]).
> 95% CIs: Patient-level bootstrap with 1,000 resamples.
> NME: Mean Euclidean landmark error normalized by ground-truth femur length.

| Approach | Row / Model | Resolution | Epochs | Train Time (s) | FL MAE [95% CI] (mm) | FL RMSE [95% CI] (mm) | FL Median [95% CI] (mm) | % Error [95% CI] | FL NME (mean+-SD) | Top-10 SqErr Share (%) | FP MAE (mm) | UCL MAE (mm, tiny) |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| Row A (U-Net EffNet-B4 1-ch Heatmap, flip TTA) | *Seeds Mean +- SD* | 384 px | 17.7 +- 2.9 | 330.4 +- 26.9 | 1.210 +- 0.051 | 3.708 +- 0.078 | 0.597 +- 0.049 | 3.11% +- 0.14% | 0.036 +- 0.001 | 90.8% +- 0.5% | 1.074 | 2.966 |
| Row A (U-Net EffNet-B4 1-ch Heatmap, flip TTA) | **3-Seed Median Ens** | 384 px | - | - | **1.147** [0.890, 1.485] | **3.540** [1.457, 5.552] | **0.610** [0.519, 0.704] | **2.94%** [2.03%, 4.29%] | **0.037 +- 0.151** | **90.5%** | **1.011** | **2.893** |
| Row B (ConvNeXt-Tiny Direct Reg, Order-Inv) | *Seeds Mean +- SD* | 384 px | 35.0 +- 0.0 | 397.1 +- 3.0 | 4.287 +- 0.415 | 7.318 +- 0.512 | 2.350 +- 0.354 | 10.81% +- 1.12% | 0.135 +- 0.006 | 51.8% +- 2.9% | 3.841 | 10.050 |
| Row B (ConvNeXt-Tiny Direct Reg, Order-Inv) | **3-Seed Median Ens** | 384 px | - | - | **4.149** [3.469, 4.981] | **7.088** [5.299, 9.116] | **2.197** [1.749, 2.658] | **10.40%** [8.45%, 13.02%] | **0.132 +- 0.183** | **51.5%** | **3.712** | **9.788** |
| Row C (U-Net EffNet-B4 512px Heatmap, flip TTA) | *Seeds Mean +- SD* | 512 px | 17.3 +- 5.4 | 507.8 +- 159.4 | 1.526 +- 0.287 | 6.097 +- 3.264 | 0.622 +- 0.010 | 3.97% +- 1.00% | 0.044 +- 0.004 | 90.6% +- 7.8% | 1.330 | 4.063 |
| Row C (U-Net EffNet-B4 512px Heatmap, flip TTA) | **3-Seed Median Ens** | 512 px | - | - | **1.210** [0.969, 1.472] | **2.957** [1.734, 3.972] | **0.632** [0.532, 0.706] | **2.97%** [2.23%, 3.84%] | **0.038 +- 0.131** | **83.3%** | **1.148** | **2.012** |


In [2]:
json_path = results_dir / 'femur_ablation_results.json'
if json_path.is_file():
    with open(json_path, 'r', encoding='utf-8') as f:
        results = json.load(f)
    print('Loaded ablation results for keys:', list(results.keys()))
    for rk in ['row_A', 'row_B', 'row_C']:
        if rk in results:
            ens = results[rk]['ensemble_all']
            ms = results[rk]['seeds_mean_sd']
            print(f'=== {results[rk]["name"]} ===')
            print(f'  Seeds Mean: MAE={ms["fl_mae_mean"]:.3f} +- {ms["fl_mae_sd"]:.3f} mm, NME={ms["fl_nme_mean"]:.3f} +- {ms["fl_nme_sd"]:.3f}')
            print(f'  Ensemble:   MAE={ens["fl_mae"]:.3f} [{ens["fl_mae_ci"][0]:.3f}, {ens["fl_mae_ci"][1]:.3f}] mm, NME={ens["fl_nme_mean"]:.3f} +- {ens["fl_nme_sd"]:.3f}')
            print(f'  Top-10 SqErr Share: {ens["top10_sq_err_share"]*100:.1f}%')
else:
    print('Results JSON not found at', json_path)

Loaded ablation results for keys: ['row_A', 'row_B', 'row_C']
=== Row A (EffNet-B4 1-ch Heatmap, 384px) ===
  Seeds Mean: MAE=1.210 +- 0.051 mm, NME=0.036 +- 0.001
  Ensemble:   MAE=1.147 [0.890, 1.485] mm, NME=0.037 +- 0.151
  Top-10 SqErr Share: 90.5%
=== Row B (ConvNeXt-Tiny Direct Reg, 384px) ===
  Seeds Mean: MAE=4.287 +- 0.415 mm, NME=0.135 +- 0.006
  Ensemble:   MAE=4.149 [3.469, 4.981] mm, NME=0.132 +- 0.183
  Top-10 SqErr Share: 51.5%
=== Row C (EffNet-B4 1-ch Heatmap, 512px) ===
  Seeds Mean: MAE=1.526 +- 0.287 mm, NME=0.044 +- 0.004
  Ensemble:   MAE=1.210 [0.969, 1.472] mm, NME=0.038 +- 0.131
  Top-10 SqErr Share: 83.3%


In [3]:
csv_path = results_dir / 'femur_predictions.csv'
if csv_path.is_file():
    df = pd.read_csv(csv_path)
    print(f'Total predictions: {len(df)} images across {df["patient"].nunique()} patients')
    print(f'Source counts: {dict(df["source"].value_counts())}')
    assert len(df) == 348, f'Expected 348 test images, got {len(df)}'
    print('Verification check passed: 348 test predictions confirmed.')
else:
    print('Predictions CSV not found at', csv_path)

Total predictions: 348 images across 266 patients
Source counts: {'FP': np.int64(323), 'UCL': np.int64(25)}
Verification check passed: 348 test predictions confirmed.
